In [1]:
import json
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

In [12]:
test_folder_path = Path("./tests")
test = "spectra"

In [14]:
with open(test_folder_path / test / f"{test}_specs.json" ) as f:
    test_specs = json.load(f)

norms_path = test_folder_path / test / f"{test}_norms.csv"
norms = pd.read_csv(norms_path)

In [73]:
def convert_to_matrices(test):
    # get test length
    test_length = test["length"]
    # init matrices to be returned
    df_straight = pd.DataFrame()
    df_reversed = pd.DataFrame()
    # iterate scales
    for scale in test["scales"]:
        # get scale label, straight items and reversed items
        scale_label, straight_items_indices, reversed_items_indices = scale
        # iterate
        for df, items_indices in [(df_straight, straight_items_indices), (df_reversed, reversed_items_indices)]:
            # create new Series with zeroes 
            zeros = pd.Series(np.zeros(test_length))
            # correct indices, items are 1-based, matrix indices are 0-based
            matrix_indices = pd.Series(items_indices).sub(1)
            # "activate" scale items
            zeros[matrix_indices] = 1
            # update straight items matrix
            df[scale_label] = zeros
    return df_straight, df_reversed

def compute_raw_scores(data, test_specs, fillna_stategy = None):
    
    # init results
    results = pd.DataFrame()
    # clone data
    answers = data.copy()
    # get scales items matrices (two separate matrices for straight and reversed items)
    df_straight, df_reversed = convert_to_matrices(test_specs)
    
    ############################################################
    # straight items
    ############################################################
    
    # compute sum of straight items
    score_straight = np.dot(answers.fillna(0), df_straight)
    # re-compute sum of straight items if we need to replace missing items with mean
    if fillna_stategy == "mean":
        # init matrix for dealing with missing items
        mean_values = (score_straight / df_straight.sum(axis=0).to_numpy()).astype('int')
        score_straight = np.dot(answers.fillna(mean_values), df_straight)

    ############################################################
    # reversed items
    ############################################################
    
    # compute amount to use for reversed items
    rev = test_specs["likert"]["min"] + test_specs["likert"]["max"]
    # compute sum of reversed items
    score_reversed = np.dot(rev - answers.fillna(rev), df_reversed)
    # re-compute sum of straight items if we need to replace missing items with mean
    if fillna_stategy == "mean":
        # init matrix for dealing with missing items
        mean_values = (score_reversed / df_reversed.sum(axis=0).to_numpy()).astype('int')
        score_reversed = np.dot(answers.fillna(mean_values), df_reversed)

    #############################################################
    # reversed items
    ############################################################
    
    # compute final results
    results.loc[:, df_straight.columns] = score_straight + score_reversed
    # return results
    return results

def compute_standard_score(s, **kwargs):
    # get kwargs
    norms, type = kwargs["norms"],  kwargs["type"]
    # return standard scores
    return norms[norms["scale"].eq(s.name)].take(s)[type].values

In [74]:
# fake data
data = pd.read_csv("data.csv")
data.head()

,i1,i2,i3,i4,i5,i6,i7,i8,i9,i10,i11,i12,i13,i14,i15,i16,i17,i18,i19,i20,i21,i22,i23,i24,i25,i26,i27,i28,i29,i30,i31,i32,i33,i34,i35,i36,i37,i38,i39,i40,i41,i42,i43,i44,i45,i46,i47,i48,i49,i50,i51,i52,i53,i54,i55,i56,i57,i58,i59,i60,i61,i62,i63,i64,i65,i66,i67,i68,i69,i70,i71,i72,i73,i74,i75,i76,i77,i78,i79,i80,i81,i82,i83,i84,i85,i86,i87,i88,i89,i90,i91,i92,i93,i94,i95,i96
0,2,NaN,4,NaN,1,2,3,4,1,3,3,3,1,3,1,3,2,4,3,2,1,2,2,3,4,2,3,2,1,3,4,3,4,3,4,3,3,3,3,3,4,1,2,4,3,4,3,2,2,3,1,3,1,2,3,1,4,2,1,2,4,3,2,2,1,4,1,2,1,1,4,3,2,1,3,3,1,3,4,3,1,3,3,4,4,1,2,1,2,2,3,4,2,1,4,2
1,4,4.0,2,3.0,2,2,3,2,3,4,1,3,3,3,4,3,2,2,3,4,3,2,1,4,4,2,2,4,1,1,4,4,3,2,1,1,3,4,1,4,3,4,1,3,1,2,4,2,2,3,2,1,4,3,4,2,1,1,4,4,4,3,3,2,2,3,1,4,1,3,3,2,3,1,3,4,4,3,1,2,4,2,3,2,3,2,3,2,4,3,1,1,1,2,4,2


In [75]:
raw_scores = compute_raw_scores(data, test_specs)
raw_scores

,dep,anx,soc,pts,alc,agg,anti,drg,psy,par,man,gra,cog,pf,sui,int,ext,ri,gpi
0,15.0,21.0,11.0,13.0,20.0,17.0,10.0,12.0,18.0,16.0,12.0,19.0,14.0,17.0,16.0,60.0,59.0,65.0,184.0
1,14.0,20.0,12.0,18.0,17.0,15.0,24.0,16.0,15.0,19.0,13.0,16.0,13.0,21.0,11.0,64.0,72.0,63.0,199.0


In [76]:
t_scores = raw_scores.apply(compute_standard_score, norms=norms, type="tscore")
t_scores

,dep,anx,soc,pts,alc,agg,anti,drg,psy,par,man,gra,cog,pf,sui,int,ext,ri,gpi
0,69,68,58,59,111,77,51,87,117,77,74,65,74,27,94,63,88,93,79
1,68,66,60,68,97,71,97,112,99,90,78,60,69,31,66,64,105,90,87
